In [58]:
!pip install imbalanced-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier

# Load the Twitter sentiment preprocessed dataset
df = pd.read_csv("../data/TweetSentiment_preprocessed.csv")

In [2]:
'''
# binary classification
df = df[df['sentiment'] != 'neutral'].copy()'''

"\n# binary classification\ndf = df[df['sentiment'] != 'neutral'].copy()"

In [3]:
# split the dataset into training and test sets using 80/20 split
X = df['text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
'''# Convert texts to TF-IDF representation
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

X_train_transformed = tfidf.fit_transform(X_train)
X_test_transformed = tfidf.transform(X_test)'''

'# Convert texts to TF-IDF representation\ntfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)\n\nX_train_transformed = tfidf.fit_transform(X_train)\nX_test_transformed = tfidf.transform(X_test)'

In [5]:
# Bag of words
bow = CountVectorizer(max_features=5000)

X_train_transformed = bow.fit_transform(X_train)
X_test_transformed = bow.transform(X_test)

In [6]:
'''# Apply random oversampler
ros = RandomOverSampler(random_state=42)

X_train_transformed, y_train = ros.fit_resample(
    X_train_transformed,
    y_train
)'''

'# Apply random oversampler\nros = RandomOverSampler(random_state=42)\n\nX_train_transformed, y_train = ros.fit_resample(\n    X_train_transformed,\n    y_train\n)'

In [7]:
# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_transformed, y_train)

# Make predictions
y_pred = nb_model.predict(X_test_transformed)

In [8]:
'''# Forward feed neural network
nn_model = MLPClassifier(
    hidden_layer_sizes=(100, ),
    activation='relu',
    max_iter=10,
    alpha = 0.1,
    random_state=42
)

nn_model.fit(X_train_transformed, y_train)
y_pred = nn_model.predict(X_test_transformed)'''

"# Forward feed neural network\nnn_model = MLPClassifier(\n    hidden_layer_sizes=(100, ),\n    activation='relu',\n    max_iter=10,\n    alpha = 0.1,\n    random_state=42\n)\n\nnn_model.fit(X_train_transformed, y_train)\ny_pred = nn_model.predict(X_test_transformed)"

In [ ]:
# Print accuracy
accuracy = accuracy_score(y_test, y_pred)
#print("Accuracy:", accuracy)

# Print metrics
print(classification_report(y_test, y_pred))
results = pd.DataFrame({
    'text': X_test,
    'actual': y_test,
    'predicted': y_pred
})

# confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred
)
print(cm)

# Error observation
errors = results[
    results['actual'] != results['predicted']
]
# Export to a csv file for observing errors
errors.to_csv(
    "../data/errors.csv",
    index=False
)

              precision    recall  f1-score   support

    negative       0.66      0.62      0.64      1556
     neutral       0.59      0.66      0.63      2223
    positive       0.73      0.66      0.70      1717

    accuracy                           0.65      5496
   macro avg       0.66      0.65      0.65      5496
weighted avg       0.66      0.65      0.65      5496

[[ 968  518   70]
 [ 403 1471  349]
 [  87  489 1141]]
